In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
from tqdm import tqdm

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()

In [ ]:
import h5py
import numpy as np
f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [77, 66, 67, 68, 69, 70, 71, 101, 102, 103]
mask_idx_list = [106] # visual processing
# mask_idx_list = [96, 97, 200] # eye movement control
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1

In [ ]:
# pv.set_jupyter_backend('static')
plotter = pv.Plotter(notebook=True)
plotter.add_volume(
    grid,
    cmap="gray",
    opacity=[0.0,0.1,0.3,0.5],
    shade=True,
    show_scalar_bar=False,
)

colors=np.mean(traces[..., valid_coordinate_i], 0)>0.13
plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='viridis',
    point_size=5,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.5)
plotter.show()

Try sbi with neural trace data as-is

In [ ]:
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE

# _ = torch.manual_seed(0)

In [ ]:
context = 4
n_d = context
horizon = 32
n_neurons = traces.shape[-1]
traces.shape[0]-context-horizon
trace_data_x = np.stack([traces[i:i+context] for i in range(0, traces.shape[0]-context-horizon, 10)])
trace_data_theta = np.stack([traces[i+context+horizon] for i in range(0, traces.shape[0]-context-horizon, 10)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::500])
trace_data_theta_train = torch.tensor(trace_data_theta[::500, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
trace_data_x_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=3.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128]

In [ ]:
n_ix = 50
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()  # for easier indexing

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context, theta_hat_i.shape[0]+context),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

plt.tight_layout()

In [ ]:
np.quantile(theta_hat_i, 0.5, axis=-1).shape
traces[:200].shape

Next, we train exactly the same network, but condition on visual input stimulus

In [ ]:
context = 8
n_d = context
horizon = 16
n_neurons = traces.shape[-1]
stimuli_x = np.where(s==1)[-1]

trace_data_x = np.stack([traces[..., valid_coordinate_i][i:i+context] for i in range(0, traces[..., valid_coordinate_i].shape[0]-context-horizon, 100)])
trace_data_theta = np.stack([traces[..., valid_coordinate_i][i+context+horizon] for i in range(0, traces[..., valid_coordinate_i].shape[0]-context-horizon, 100)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*trace_data_x.shape[2], context)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x = np.stack([traces[..., valid_coordinate_i][i:i+context] for i in range(0, traces[..., valid_coordinate_i].shape[0]-context-horizon, 100)])

In [ ]:
trace_data_x.shape

In [ ]:
plt.plot(stimuli_x[:1000])
plt.plot(np.cos(np.arange(1000)/(100/(3/2*np.pi))))

In [ ]:
stimuli_x_data = np.stack([stimuli_x[i:i+context] for i in range(0, stimuli_x.shape[0]-context-horizon, 100)])

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::10])
trace_data_theta_train = torch.tensor(trace_data_theta[::10, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=3.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
for neuron_ix in range(20000, 20010):
  predicted_trace_i = []
  theta_hat_i = []
  for j in tqdm(range(0, 200)):
    x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
    theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
    theta_hat_i.append(theta_hat)
    predicted_trace_i.append(theta_hat.mean())
  theta_hat_i = np.array(theta_hat_i).squeeze(-1)
  plt.figure(figsize=(10, 1), dpi=500)
  plt.plot(traces[:200+context, neuron_ix], 'k')
  plt.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
  plt.fill_between(
    np.arange(context, theta_hat_i.shape[0]+context),
    np.quantile(theta_hat_i, 0.05, axis=-1),
    np.quantile(theta_hat_i, 0.95, axis=-1),
    color='red',
    alpha=0.1,
    linewidth=0,
  );

In [ ]:
for neuron_ix in valid_coordinate_i[1400:1410]:
  predicted_trace_i = []
  theta_hat_i = []
  for j in tqdm(range(0, 200)):
    x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
    theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
    theta_hat_i.append(theta_hat)
    predicted_trace_i.append(theta_hat.mean())
  theta_hat_i = np.array(theta_hat_i).squeeze(-1)
  plt.figure(figsize=(10, 1), dpi=500)
  plt.plot(traces[:200+context, neuron_ix], 'k')
  plt.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
  plt.fill_between(
    np.arange(context, theta_hat_i.shape[0]+context),
    np.quantile(theta_hat_i, 0.05, axis=-1),
    np.quantile(theta_hat_i, 0.95, axis=-1),
    color='red',
    alpha=0.1,
    linewidth=0,
  );